### Конкурентность vs Параллелизм, GIL

In [ ]:
import time
import threading

def cpu_bound(n: int) -> int:
    return sum(i * i for i in range(n))

N = 150_000_000
CHUNKS = 4

start = time.time()
for _ in range(CHUNKS):
    cpu_bound(N // CHUNKS)
sequential_time = time.time() - start

start = time.time()
threads = [threading.Thread(target=cpu_bound, args=(N // CHUNKS,)) for _ in range(CHUNKS)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threading_time = time.time() - start

print(f"Последовательно : {sequential_time:.2f} сек")
print(f"Threading       : {threading_time:.2f} сек")

In [ ]:
import os
cores = os.cpu_count()
print(f"Доступно ядер CPU: {cores}")

### Будущее GIL: PEP 703

In [ ]:
import sys

gil_enabled = getattr(sys, "_is_gil_enabled", None)
if gil_enabled is not None:
    print(f"GIL включён: {gil_enabled()}")
else:
    print("Эта сборка Python не поддерживает free-threaded режим (PEP 703 недоступен).")
print(f"Версия интерпретатора: {sys.version}")

### Базовый синтаксис Threading

In [ ]:
def download(name: str, delay: float) -> None:
    print(f"Начало загрузки {name}")
    time.sleep(delay)  # блокирующий syscall - освобождает GIL
    print(f"Загрузка {name} завершена")

threads = [threading.Thread(target=download, args=(f"файл_{i}", 1)) for i in range(5)]
print(f"threads: {threads}")

start = time.time()
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nВсего: {time.time() - start:.1f} сек")  # потоки перекрываются

### Coroutine, Task, Future и базовый синтаксис

In [ ]:
import asyncio

async def download_async(name: str, delay: float) -> None:
    print(f"Начало загрузки {name}")
    await asyncio.sleep(delay)  # не блокирует event loop
    print(f"Загрузка {name} завершена")

start = time.time()
await asyncio.gather(
    download_async("файл_1", 1),
    download_async("файл_2", 1),
    download_async("файл_3", 1),
    download_async("файл_4", 1),
    download_async("файл_5", 1),
)
print(f"\nВсего: {time.time() - start:.1f} сек")

In [ ]:
async def bad(delay):
    time.sleep(delay)   # блокирует event loop, включая другие задачи

async def good(delay):
    await asyncio.sleep(delay)   # отдаёт управление циклу

# Демонстрация: "плохие" корутины выполняются последовательно, будто это обычные функции
start = time.time()
await asyncio.gather(bad(0.5), bad(0.5), bad(0.5))
print(f"bad():  {time.time() - start:.2f} сек  <- перекрылись, выполнились последовательно")

start = time.time()
await asyncio.gather(good(0.5), good(0.5), good(0.5))
print(f"good(): {time.time() - start:.2f} сек  <- перекрылись, как и задумано")

### Ограничение конкурентности: `Semaphore`

In [ ]:
async def fetch_price(ticker: str, semaphore: asyncio.Semaphore) -> dict:
    print(f"{ticker}: семафор (счетчик = {semaphore._value})")
    async with semaphore:
        print(f"{ticker}: Задача принята, семафор (счетчик = {semaphore._value})")
        await asyncio.sleep(0.2)  # имитация задержки
        result = {"ticker": ticker, "price": round(100 + hash(ticker) % 50, 2)}
        print(f"{ticker}: Выход, семафор (счетчик = {semaphore._value})")
        return result

async def fetch_all(tickers: list[str]) -> list[dict]:
    semaphore = asyncio.Semaphore(10)  # не более 10 одновременных "запросов"
    tasks = [fetch_price(t, semaphore) for t in tickers]
    return await asyncio.gather(*tasks)

tickers = [f"TICK{i}" for i in range(100)]

start = time.time()
results = await fetch_all(tickers)
print(f"Получено {len(results)} котировок за {time.time() - start:.2f} сек")
print(results)

### `gather` vs `as_completed` vs `TaskGroup`

In [ ]:
async def worker(n: int, delay: float) -> int:
    await asyncio.sleep(delay)
    return n

# gather - порядок результатов сохраняется
results = await asyncio.gather(worker(1, 0.3), worker(2, 0.1), worker(3, 0.2))
print("gather (порядок исходный):", results)

# as_completed - готовые результаты приходят в порядке завершения, а не запуска
print("as_completed (порядок готовности):", end=" ")
for coro in asyncio.as_completed([worker(1, 0.3), worker(2, 0.1), worker(3, 0.2)]):
    print(await coro, end=" ")
print()

### Структурная конкурентность

In [ ]:
# TaskGroup (3.11+) - структурная конкурентность
results = []
async with asyncio.TaskGroup() as tg:
    t1 = tg.create_task(worker(1, 0.1))
    t2 = tg.create_task(worker(2, 0.1))

results = [t1.result(), t2.result()]
print("TaskGroup результаты:", results)

In [ ]:
async def failing_worker():
    await asyncio.sleep(0.1)
    raise ValueError("Что-то пошло не так")

async def long_worker():
    try:
        await asyncio.sleep(5)
        return "не должно сюда дойти"
    except asyncio.CancelledError:
        print("long_worker: получена отмена - TaskGroup среагировал на ошибку соседа")
        raise

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(failing_worker())
        tg.create_task(long_worker())
except ValueError as eg:
    print(f"Поймано исключений в группе: {len(eg.exceptions)} -> {eg.exceptions[0]}")

In [ ]:
task1 = asyncio.create_task(failing_worker())
task2 = asyncio.create_task(long_worker())
    
results = await asyncio.gather(task1, task2, return_exceptions=True)
print(results)

### Асинхронные контекстные менеджеры

In [ ]:
class AsyncDatabaseConnection:
    async def __aenter__(self):
        print("Открываем соединение...")
        await asyncio.sleep(0.1)  # имитация асинхронного подключения
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("Закрываем соединение...")
        await asyncio.sleep(0.1)  # имитация асинхронного закрытия

    async def query(self, sql: str) -> list:
        await asyncio.sleep(0.2)
        return [{"result": "данные"}]

async with AsyncDatabaseConnection() as conn:
    data = await conn.query("SELECT * FROM prices")
    print("Результат запроса:", data)

### Асинхронные генераторы

In [ ]:
async def stream_prices(ticker: str):
    for i in range(5):
        await asyncio.sleep(0.2)  # имитация поступления новой цены
        yield {"ticker": ticker, "price": 100 + i}

async for price in stream_prices("AAPL"):
    print(price)

```text
Блоки кода ниже представлены в виде отдельных .py файлов, вы можете запускать их через python или скопировав в юпитер
```

### Многопроцессорность: Базовая модель `Process`

### Atomic

### `Pool` и `ProcessPoolExecutor`

### Издержки сериализации

### Обход сериализации: `shared_memory`